# Improved Central Limit Theorem and Bootstrap Approximations for Linear Stochastic Approximation

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from scipy.stats import norm

# Data

In [20]:
def generate_data(n_trajs, n, theta_true, d=5, sigma=0.02, seed=2345, use_tqdm=True):
    rng = np.random.default_rng(seed)
    all_data = []
    traj_arr = range(n_trajs)
    if use_tqdm:
        traj_arr = tqdm(range(n_trajs))
    for i_traj in traj_arr:
        X = rng.uniform(-1, 1, size=(n, d))
        y = X @ theta_true + rng.normal(0, sigma, size=n)
        all_data.append(
            (X, y)
        )
    return all_data

In [3]:
n = 50000 + 1
d = 5
K = 256
theta_true = np.array([1, 1, -0.1, -0.1, -0.1])

In [4]:
n_trajs = 1024
all_data = generate_data(n_trajs, n, theta_true)

  0%|          | 0/1024 [00:00<?, ?it/s]

# Bootstrap

In [5]:
def get_boot_alphas(alphas, k, seed=1111, a=2, b=2):
    rng = np.random.default_rng(seed)
    w = rng.beta(a, b, size=(k, len(alphas)))

    mean_w = a / (a + b)
    var_w = (a * b) / ((a + b) ** 2 * (a + b + 1))
    std_w = np.sqrt(var_w)

    alphas_boot = alphas * (1 + (w - mean_w) / std_w)

    return alphas_boot

class LSA:
    def __init__(self, params):
        self.n = params['n']
        self.d = params['d']
        self.gamma = params['gamma']
        self.alphas = params['c0'] / (params['k0'] + np.arange(1, self.n + 1)) ** self.gamma

        self.A = {}
        self.b = {}

        self.K = params['K']

        self.theta_bar_save_every = dict()

        self.theta_history = None
        self.theta_bar_history = None

    def fit(self, data, theta_zero, save_every=10000, burn_in=100, save_all_history=False):
        self.A = data['A']
        self.b = data['b']

        self.theta_bar_save_every = self.count_traj(self.alphas, theta_zero=theta_zero, save_every=save_every, burn_in=burn_in, save_all_history=save_all_history)

    def count_traj(self, alphas, regime='real_world', save_every=10000):
        pass

    def get_boot_sample(self, theta_zero=None, seed=1234, save_every=10000, burn_in=100):
        alphas_boot = get_boot_alphas(self.alphas, self.K, seed)
        theta_boot = np.zeros((self.K, self.d)) + theta_zero
        return self.count_traj(alphas_boot, regime='bootstrap', theta_zero=theta_boot, save_every=save_every, burn_in=burn_in)

class LinRegLSA(LSA):
    def count_traj(self, alphas, regime='real_world', theta_zero=None, save_every=10000, burn_in=100, save_all_history=False):
        if theta_zero is None:
            theta_zero = np.zeros(self.d)
        theta = np.copy(theta_zero)
        theta_mean = np.zeros_like(theta_zero)

        theta_bar_save_every = dict()

        len_from_burn_in = 0

        if regime == 'real_world' and save_all_history:
            self.theta_history = np.zeros((self.n, self.d))
            self.theta_history[0] = theta
            self.theta_bar_history = np.zeros((self.n, self.d))
            self.theta_bar_history[0] = theta

        for i in range(1, self.n):
            if regime == 'real_world':
                theta = theta - 2 * alphas[i] * self.A[i] * (self.A[i] @ theta - self.b[i])
                if save_all_history:
                    self.theta_history[i] = theta
                    self.theta_bar_history[i] = (i * self.theta_bar_history[i - 1] + theta) / (i + 1)
            else:
                pred = theta @ self.A[i]
                error = pred - self.b[i]
                grad = error[:, None] * self.A[i]
                grad *= alphas[:, i][:, None]
                theta = theta - 2 * grad

            if i >= burn_in:
                theta_mean = (len_from_burn_in * theta_mean + theta) / (len_from_burn_in + 1)
                len_from_burn_in += 1

            if i % save_every == 0:
                theta_bar_save_every[i] = np.copy(theta_mean)
        return theta_bar_save_every

# Empirical Quantiles

In [18]:
def count_coverage_proba(theta_true, all_data, params, conf=5, n_trajs=100, seed=3456, save_every=10000, use_tqdm=True):
    proj = np.random.uniform(-1, 1, size=params['d'])
    proj /= np.linalg.norm(proj)

    theta_true_proj = proj @ theta_true

    last_history_index = params['n'] // save_every * save_every
    cov_probas = dict([(step, 0) for step in range(save_every, last_history_index + 1, save_every)])

    traj_arr = range(n_trajs)
    if use_tqdm:
        traj_arr = tqdm(range(n_trajs))

    for i_traj in traj_arr:
        X, y = all_data[i_traj]

        data = {
            'A': X,
            'b': y
        }

        model = LinRegLSA(params)
        model.fit(data, theta_true)


        theta_bar_history = model.theta_bar_save_every
        theta_boot_bar_history = model.get_boot_sample(theta_true, seed=seed + i_traj, save_every=save_every)


        for step, theta_bar_boot_cur in theta_boot_bar_history.items():
            theta_bar_cur = theta_bar_history[step]

            theta_bar_proj = proj @ theta_bar_cur
            theta_bar_boot_proj = theta_bar_boot_cur @ proj
            boot_mean = np.mean(theta_bar_boot_proj)

            centered_quantiles = np.percentile(theta_bar_boot_proj, [conf / 2, 100 - conf / 2])

            q_low, q_high = centered_quantiles

            if theta_true_proj >= q_low and theta_true_proj <= q_high:
                cov_probas[step] += 1 / n_trajs

        min_cov = np.min(list(cov_probas.values()))
        max_cov = np.max(list(cov_probas.values()))
        argmins = sorted([step for step in cov_probas.keys() if cov_probas[step] == min_cov])
        argmaxs = sorted([step for step in cov_probas.keys() if cov_probas[step] == max_cov])
        if use_tqdm:
            traj_arr.set_postfix(min_cov=f"{min_cov * n_trajs / (i_traj + 1):.3f}",
                            max_cov=f"{max_cov * n_trajs / (i_traj + 1):.3f}",
                            argmins=f"{argmins}",
                            argmaxes=f"{argmaxs}")
    return cov_probas

In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.85,
    'c0': 200,
    'k0': 20000,
}

count_coverage_proba(theta_true, all_data, params, n_trajs=n_trajs)

  0%|          | 0/1024 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.9,
    'c0': 200,
    'k0': 20000,
}

count_coverage_proba(theta_true, all_data, params, n_trajs=n_trajs)

  0%|          | 0/1024 [00:00<?, ?it/s]

{10000: 0.94921875,
 20000: 0.953125,
 30000: 0.9443359375,
 40000: 0.947265625,
 50000: 0.9365234375,
 60000: 0.94921875,
 70000: 0.94140625,
 80000: 0.947265625,
 90000: 0.955078125,
 100000: 0.953125}

In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.95,
    'c0': 200,
    'k0': 20000,
}

count_coverage_proba(theta_true, all_data, params, n_trajs=n_trajs)

  0%|          | 0/1024 [00:00<?, ?it/s]

{10000: 0.9541015625,
 20000: 0.951171875,
 30000: 0.955078125,
 40000: 0.953125,
 50000: 0.951171875,
 60000: 0.951171875,
 70000: 0.9521484375,
 80000: 0.947265625,
 90000: 0.947265625,
 100000: 0.939453125}

# Standard Deviation-Based Confidence Intervals

In [ ]:
def count_coverage_proba_normal(theta_true, all_data, params, conf=5, n_trajs=100, seed=3456, save_every=10000,
                                burn_in=100):
    proj = np.random.uniform(-1, 1, size=5)
    proj /= np.linalg.norm(proj)
    theta_true_proj = proj @ theta_true

    last_history_index = params['n'] // save_every * save_every
    cov_probas = dict([(step, 0) for step in range(save_every, last_history_index + 1, save_every)])

    pbar = tqdm(range(n_trajs))

    for i_traj in pbar:
        X, y = all_data[i_traj]

        data = {
            'A': X,
            'b': y
        }

        model = LinRegLSA(params)
        model.fit(data, theta_true, burn_in=burn_in)


        theta_bar_history = model.theta_bar_save_every
        theta_boot_bar_history = model.get_boot_sample(theta_true, seed=seed + i_traj, save_every=save_every, burn_in=burn_in)


        for step, theta_bar_boot_cur in theta_boot_bar_history.items():
            theta_bar_cur = theta_bar_history[step]

            m = step - burn_in

            theta_bar_proj = proj @ theta_bar_cur
            theta_bar_boot_proj = np.sqrt(m) * (theta_bar_boot_cur @ proj - theta_bar_proj)

            var = np.var(theta_bar_boot_proj)
            std = np.sqrt(var)

            alpha = conf / 100
            z = norm.ppf(1 - alpha/2)

            q_low = theta_bar_proj - z * std / np.sqrt(m)
            q_high = theta_bar_proj + z * std / np.sqrt(m)

            if q_low <= theta_true_proj <= q_high:
                cov_probas[step] += 1 / n_trajs
        min_cov = np.min(list(cov_probas.values()))
        max_cov = np.max(list(cov_probas.values()))
        argmins = sorted([step for step in cov_probas.keys() if cov_probas[step] == min_cov])
        argmaxs = sorted([step for step in cov_probas.keys() if cov_probas[step] == max_cov])
        pbar.set_postfix(min_cov=f"{min_cov * n_trajs / (i_traj + 1):.3f}",
                         max_cov=f"{max_cov * n_trajs / (i_traj + 1):.3f}",
                         argmins=f"{argmins}",
                         argmaxes=f"{argmaxs}")
    return cov_probas

In [ ]:
params = {
    'n': 50000 + 1,
    'd': 5,
    'K': 256,
    'gamma': 0.85,
    'c0': 200,
    'k0': 20000,
}

count_coverage_proba_normal(theta_true, all_data, params, n_trajs=n_trajs)

  0%|          | 0/1024 [00:00<?, ?it/s]

{10000: 0.9619140625,
 20000: 0.958984375,
 30000: 0.9521484375,
 40000: 0.94921875,
 50000: 0.947265625}

In [ ]:
params = {
    'n': 50000 + 1,
    'd': 5,
    'K': 256,
    'gamma': 0.9,
    'c0': 200,
    'k0': 20000,
}

count_coverage_proba_normal(theta_true, all_data, params, n_trajs=n_trajs)

  0%|          | 0/1024 [00:00<?, ?it/s]

{10000: 0.9609375,
 20000: 0.96484375,
 30000: 0.9580078125,
 40000: 0.953125,
 50000: 0.9609375}

In [ ]:
params = {
    'n': 50000 + 1,
    'd': 5,
    'K': 256,
    'gamma': 0.95,
    'c0': 200,
    'k0': 20000,
}

count_coverage_proba_normal(theta_true, all_data, params, n_trajs=n_trajs)

  0%|          | 0/1024 [00:00<?, ?it/s]

{10000: 0.9462890625,
 20000: 0.9541015625,
 30000: 0.955078125,
 40000: 0.947265625,
 50000: 0.94921875}

# Overlapping Batch Mean Estimator

In [ ]:
def batch_mean(theta_hat_history, b_n):
    N_iters, N_s = theta_hat_history.shape
    BM_V = np.empty((N_iters - b_n + 1, N_s), dtype=theta_hat_history.dtype)

    cur_bm = theta_hat_history[0:b_n, :].mean(axis=0)
    for i in range(N_iters - b_n):
        BM_V[i, :] = cur_bm
        cur_bm += (theta_hat_history[b_n + i, :] - theta_hat_history[i, :]) / b_n
    BM_V[-1, :] = cur_bm
    return BM_V

def cov_prob_3(all_data, params, conf, b_n, n_trajs, i):
    rng = np.random.default_rng(12345)
    u = rng.normal(size=params['d'])
    u /= np.linalg.norm(u)
    theta_in_interval = np.zeros(n_trajs, dtype=bool)
    for i_traj in tqdm(range(n_trajs), desc=f"Computing coverage for b_n = {b_n}"):
        X, y = all_data[i_traj]

        data = {
            'A': X,
            'b': y
        }

        model = LinRegLSA(params)
        model.fit(data, theta_true, save_all_history=True)

        theta_hat_history = model.theta_history
        theta_bar_history = model.theta_bar_history

        BM_V = batch_mean(theta_hat_history, b_n)

        hat_sigma =  (((BM_V - theta_bar_history[-1, :]) @ u) ** 2).sum() * (b_n / (50001-b_n))
        std = np.sqrt(hat_sigma)
        alpha = 1 - conf / 100
        z = norm.ppf(1 - alpha / 2)

        lower = theta_bar_history[(i + 1) * 10000, :] @ u - z * std / np.sqrt((i + 1) * 10000 + 1)
        upper = theta_bar_history[(i + 1) * 10000, :] @ u + z * std / np.sqrt((i + 1) * 10000 + 1)

        true_proj = theta_true @ u
        theta_in_interval[i_traj] = int((lower <= true_proj <= upper))
    coverage = np.mean(theta_in_interval)
    return coverage

In [ ]:
def get_bn(C, params):
    b_n_arr = [int(C * np.log(params['n'])),
                int(C * params['n'] ** (1/3)),
                int(C * params['n'] ** (1/2))]
    return b_n_arr

In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.85,
    'c0': 200,
    'k0': 20000,
}

C = 3
b_n_arr = get_bn(C, params)

for b_n in b_n_arr:
    print(b_n, cov_prob_3(all_data, params, 95, b_n, n_trajs, 4))

Computing coverage for b_n = 32:   0%|          | 0/1024 [00:00<?, ?it/s]

32 0.6435546875


Computing coverage for b_n = 110:   0%|          | 0/1024 [00:00<?, ?it/s]

110 0.84375


Computing coverage for b_n = 670:   0%|          | 0/1024 [00:00<?, ?it/s]

670 0.93359375


In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.85,
    'c0': 200,
    'k0': 20000,
}

C = 4
b_n_arr = get_bn(C, params)

for b_n in b_n_arr:
    print(b_n, cov_prob_3(all_data, params, 95, b_n, n_trajs, 4))

Computing coverage for b_n = 43:   0%|          | 0/1024 [00:00<?, ?it/s]

43 0.6962890625


Computing coverage for b_n = 147:   0%|          | 0/1024 [00:00<?, ?it/s]

147 0.8701171875


Computing coverage for b_n = 894:   0%|          | 0/1024 [00:00<?, ?it/s]

894 0.9345703125


In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.85,
    'c0': 200,
    'k0': 20000,
}

C = 5
b_n_arr = get_bn(C, params)

for b_n in b_n_arr:
    print(b_n, cov_prob_3(all_data, params, 95, b_n, n_trajs, 4))

Computing coverage for b_n = 54:   0%|          | 0/1024 [00:00<?, ?it/s]

54 0.740234375


Computing coverage for b_n = 184:   0%|          | 0/1024 [00:00<?, ?it/s]

184 0.888671875


Computing coverage for b_n = 1118:   0%|          | 0/1024 [00:00<?, ?it/s]

1118 0.9365234375


In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.85,
    'c0': 200,
    'k0': 20000,
}

C = 7
b_n_arr = get_bn(C, params)

for b_n in b_n_arr:
    print(b_n, cov_prob_3(all_data, params, 95, b_n, n_trajs, 4))

Computing coverage for b_n = 75:   0%|          | 0/1024 [00:00<?, ?it/s]

75 0.7880859375


Computing coverage for b_n = 257:   0%|          | 0/1024 [00:00<?, ?it/s]

257 0.90625


Computing coverage for b_n = 1565:   0%|          | 0/1024 [00:00<?, ?it/s]

1565 0.931640625


In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.85,
    'c0': 200,
    'k0': 20000,
}

C = 10
b_n_arr = get_bn(C, params)

for b_n in b_n_arr:
    print(b_n, cov_prob_3(all_data, params, 95, b_n, n_trajs, 4))

Computing coverage for b_n = 108:   0%|          | 0/1024 [00:00<?, ?it/s]

108 0.841796875


Computing coverage for b_n = 368:   0%|          | 0/1024 [00:00<?, ?it/s]

368 0.919921875


Computing coverage for b_n = 2236:   0%|          | 0/1024 [00:00<?, ?it/s]

2236 0.9267578125


In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.95,
    'c0': 200,
    'k0': 20000,
}

C = 3
b_n_arr = get_bn(C, params)

for b_n in b_n_arr:
    print(b_n, cov_prob_3(all_data, params, 95, b_n, n_trajs, 4))

Computing coverage for b_n = 32:   0%|          | 0/1024 [00:00<?, ?it/s]

32 0.4365234375


Computing coverage for b_n = 110:   0%|          | 0/1024 [00:00<?, ?it/s]

110 0.6845703125


Computing coverage for b_n = 670:   0%|          | 0/1024 [00:00<?, ?it/s]

In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.95,
    'c0': 200,
    'k0': 20000,
}


C = 3
b_n_arr = get_bn(C, params)
print(b_n, cov_prob_3(all_data, params, 95, b_n_arr[2], n_trajs, 4))

Computing coverage for b_n = 670:   0%|          | 0/1024 [00:00<?, ?it/s]

1118 0.8916015625


In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.95,
    'c0': 200,
    'k0': 20000,
}

C = 5
b_n_arr = get_bn(C, params)

for b_n in b_n_arr:
    print(b_n, cov_prob_3(all_data, params, 95, b_n, n_trajs, 4))

Computing coverage for b_n = 54:   0%|          | 0/1024 [00:00<?, ?it/s]

54 0.53515625


Computing coverage for b_n = 184:   0%|          | 0/1024 [00:00<?, ?it/s]

184 0.7626953125


Computing coverage for b_n = 1118:   0%|          | 0/1024 [00:00<?, ?it/s]

In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.95,
    'c0': 200,
    'k0': 20000,
}

C = 5
b_n_arr = get_bn(C, params)
print(b_n_arr[2], cov_prob_3(all_data, params, 95, b_n_arr[2], n_trajs, 4))

Computing coverage for b_n = 1118:   0%|          | 0/1024 [00:00<?, ?it/s]

1118 0.9130859375


In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.95,
    'c0': 200,
    'k0': 20000,
}

C = 7
b_n_arr = get_bn(C, params)

for b_n in b_n_arr:
    print(b_n, cov_prob_3(all_data, params, 95, b_n, n_trajs, 4))

Computing coverage for b_n = 75:   0%|          | 0/1024 [00:00<?, ?it/s]

75 0.6142578125


Computing coverage for b_n = 257:   0%|          | 0/1024 [00:00<?, ?it/s]

257 0.81640625


Computing coverage for b_n = 1565:   0%|          | 0/1024 [00:00<?, ?it/s]

1565 0.9150390625


In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.95,
    'c0': 200,
    'k0': 20000,
}

C = 10
b_n_arr = get_bn(C, params)

for b_n in b_n_arr:
    print(b_n, cov_prob_3(all_data, params, 95, b_n, n_trajs, 4))

Computing coverage for b_n = 108:   0%|          | 0/1024 [00:00<?, ?it/s]

108 0.6826171875


Computing coverage for b_n = 368:   0%|          | 0/1024 [00:00<?, ?it/s]

368 0.8603515625


Computing coverage for b_n = 2236:   0%|          | 0/1024 [00:00<?, ?it/s]

2236 0.9150390625


## Confidence Intervals Lengths

In [ ]:
import numpy as np
from tqdm import tqdm

def count_stats(theta_true, all_data, params, conf=5, n_trajs=100, seed=3456, save_every=10000):
    proj = np.random.uniform(-1, 1, size=5)
    proj /= np.linalg.norm(proj)

    last_history_index = params['n'] // save_every * save_every
    steps = list(range(save_every, last_history_index + 1, save_every))

    # Словари для накопительной статистики (онлайн-вычисление)
    sums = {step: 0.0 for step in steps}
    sq_sums = {step: 0.0 for step in steps}

    pbar = tqdm(range(n_trajs))

    for i_traj in pbar:
        X, y = all_data[i_traj]
        model = LinRegLSA(params)
        model.fit({'A': X, 'b': y}, theta_true)

        theta_boot_bar_history = model.get_boot_sample(theta_true, seed=seed + i_traj, save_every=save_every)

        for step in steps:
            theta_bar_boot_cur = theta_boot_bar_history[step]
            theta_bar_boot_proj = np.dot(theta_bar_boot_cur, proj)

            q_low, q_high = np.percentile(theta_bar_boot_proj, [conf / 2, 100 - conf / 2])
            length = q_high - q_low

            sums[step] += length
            sq_sums[step] += length**2

        last_step = steps[-1]
        n = i_traj + 1
        curr_mean = sums[last_step] / n
        curr_std = np.sqrt(sq_sums[last_step] / n - (curr_mean ** 2))

        pbar.set_postfix(mean=f"{curr_mean:.6f}", std=f"{curr_std:.6f}")

    results = {}
    for step in steps:
        mean = sums[step] / n_trajs
        std = np.sqrt(sq_sums[step] / n_trajs - (mean**2))
        results[step] = {'mean_length': mean, 'std_length': std}

    return results

In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.5,
    'c0': 45,
    'k0': 20000,
}

count_stats(theta_true, all_data, params, n_trajs=n_trajs, save_every=10000)

100%|██████████| 1024/1024 [1:02:03<00:00,  3.64s/it, mean=325.329322, std=3453.149730]


{10000: {'mean_length': np.float64(1639.6584819585921),
  'std_length': np.float64(17403.86067270341)},
 20000: {'mean_length': np.float64(815.750873615818),
  'std_length': np.float64(8658.641521751826)},
 30000: {'mean_length': np.float64(542.933633385582),
  'std_length': np.float64(5762.871623850551)},
 40000: {'mean_length': np.float64(406.8634536413105),
  'std_length': np.float64(4318.579086664407)},
 50000: {'mean_length': np.float64(325.32932232117116),
  'std_length': np.float64(3453.149730484221)}}

In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.5,
    'c0': 10,
    'k0': 20000,
}

count_stats(theta_true, all_data, params, n_trajs=n_trajs, save_every=10000)

100%|██████████| 1024/1024 [37:04<00:00,  2.17s/it, mean=0.000693, std=0.000041]


{10000: {'mean_length': np.float64(0.0016329873590931734),
  'std_length': np.float64(9.726226638121355e-05)},
 20000: {'mean_length': np.float64(0.001133472513591159),
  'std_length': np.float64(6.476917142739752e-05)},
 30000: {'mean_length': np.float64(0.0009113955840826274),
  'std_length': np.float64(5.3325538468762975e-05)},
 40000: {'mean_length': np.float64(0.0007827326412136357),
  'std_length': np.float64(4.5466095713421734e-05)},
 50000: {'mean_length': np.float64(0.0006934703858217553),
  'std_length': np.float64(4.123441309891309e-05)}}

In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.7,
    'c0': 200,
    'k0': 20000,
}

count_stats(theta_true, all_data, params, n_trajs=n_trajs, save_every=10000)

100%|██████████| 1024/1024 [54:08<00:00,  3.17s/it, mean=0.000947, std=0.000055]


{10000: {'mean_length': np.float64(0.002725654444533778),
  'std_length': np.float64(0.00016405988806928956)},
 20000: {'mean_length': np.float64(0.0017396898733556673),
  'std_length': np.float64(9.90504497768431e-05)},
 30000: {'mean_length': np.float64(0.0013316085795756537),
  'std_length': np.float64(7.702735631280651e-05)},
 40000: {'mean_length': np.float64(0.0010987959080642781),
  'std_length': np.float64(6.136728823436413e-05)},
 50000: {'mean_length': np.float64(0.0009471742973473865),
  'std_length': np.float64(5.504856052607948e-05)}}

In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.75,
    'c0': 200,
    'k0': 20000,
}

count_stats(theta_true, all_data, params, n_trajs=n_trajs, save_every=10000)

100%|██████████| 1024/1024 [54:13<00:00,  3.18s/it, mean=0.000752, std=0.000045]


{10000: {'mean_length': np.float64(0.0018851015512127167),
  'std_length': np.float64(0.0001126817221715411)},
 20000: {'mean_length': np.float64(0.0012751121164252754),
  'std_length': np.float64(7.585458207411727e-05)},
 30000: {'mean_length': np.float64(0.0010085450738729633),
  'std_length': np.float64(5.991952944265982e-05)},
 40000: {'mean_length': np.float64(0.0008549020895983908),
  'std_length': np.float64(5.08931762029793e-05)},
 50000: {'mean_length': np.float64(0.0007520984515333215),
  'std_length': np.float64(4.496160370583268e-05)}}

In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.85,
    'c0': 200,
    'k0': 20000,
}

count_stats(theta_true, all_data, params, n_trajs=n_trajs, save_every=10000)

100%|██████████| 1024/1024 [54:18<00:00,  3.18s/it, mean=0.000643, std=0.000037]


{10000: {'mean_length': np.float64(0.0014949248472031754),
  'std_length': np.float64(8.804437087062914e-05)},
 20000: {'mean_length': np.float64(0.0010409576758086741),
  'std_length': np.float64(5.995103675911951e-05)},
 30000: {'mean_length': np.float64(0.0008397629606226965),
  'std_length': np.float64(4.914177722285154e-05)},
 40000: {'mean_length': np.float64(0.0007218848100192792),
  'std_length': np.float64(4.084935370251786e-05)},
 50000: {'mean_length': np.float64(0.0006425857963031269),
  'std_length': np.float64(3.6531206942465795e-05)}}

In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.9,
    'c0': 200,
    'k0': 20000,
}

count_stats(theta_true, all_data, params, n_trajs=n_trajs, save_every=10000)

100%|██████████| 1024/1024 [54:14<00:00,  3.18s/it, mean=0.000622, std=0.000035]


{10000: {'mean_length': np.float64(0.0014258520396376651),
  'std_length': np.float64(8.408791888415987e-05)},
 20000: {'mean_length': np.float64(0.001002597799005672),
  'std_length': np.float64(5.828861093377299e-05)},
 30000: {'mean_length': np.float64(0.0008103675288273369),
  'std_length': np.float64(4.6039912322597166e-05)},
 40000: {'mean_length': np.float64(0.0006974862495463026),
  'std_length': np.float64(4.020107544497391e-05)},
 50000: {'mean_length': np.float64(0.0006217859733169686),
  'std_length': np.float64(3.53558707701874e-05)}}

In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.95,
    'c0': 200,
    'k0': 20000,
}

count_stats(theta_true, all_data, params, n_trajs=n_trajs, save_every=10000)

100%|██████████| 1024/1024 [1:05:41<00:00,  3.85s/it, mean=0.000612, std=0.000035]


{10000: {'mean_length': np.float64(0.0013876966140872644),
  'std_length': np.float64(8.046638857409677e-05)},
 20000: {'mean_length': np.float64(0.0009755350660987793),
  'std_length': np.float64(5.630426080913499e-05)},
 30000: {'mean_length': np.float64(0.0007940396134794578),
  'std_length': np.float64(4.5075427073674485e-05)},
 40000: {'mean_length': np.float64(0.0006852129929823873),
  'std_length': np.float64(4.104494590211728e-05)},
 50000: {'mean_length': np.float64(0.000611518572758199),
  'std_length': np.float64(3.466428527297553e-05)}}

## Time complexity

In [27]:
import pandas as pd
import numpy as np
from time import time
from tqdm import tqdm

d_array = [2, 8, 32, 64]
K_arr = [1, 4, 16, 64, 256]

df = pd.DataFrame(index=d_array, columns=K_arr)

n_reps = 50

for cur_d in tqdm(df.index, desc="Outer loop (d)", position=0):
    theta_true = np.random.normal(size=cur_d)
    data = generate_data(n_reps, n, theta_true, d=cur_d, use_tqdm=False)

    for cur_k in tqdm(df.columns, desc="Inner loop (n)", position=1, leave=False):
        params = {
            'n': n,
            'd': cur_d,
            'K': cur_k,
            'gamma': 0.85,
            'c0': 200,
            'k0': 20000,
        }

        time_result = 0

        time_arr = []
        for i in range(n_reps):
            start = time()
            count_coverage_proba(theta_true, [data[i]], params, n_trajs=1, save_every=params['n'] - 1, use_tqdm=False)
            time_arr.append(time() - start)

        df.loc[cur_d, cur_k] = f"{np.mean(time_arr):.5f} +- {np.std(time_arr):.5f}"

Outer loop (d): 100%|██████████| 4/4 [36:24<00:00, 546.12s/it]


In [28]:
print(df)

                   1                   4                   16   \
2   1.25733 +- 0.28482  1.30355 +- 0.27544  1.34947 +- 0.26364   
8   1.23457 +- 0.25789  1.29377 +- 0.26217  1.42587 +- 0.35746   
32  1.27686 +- 0.27602  1.33521 +- 0.26708  1.58511 +- 0.25277   
64  1.26433 +- 0.26283  1.41814 +- 0.25509  1.73276 +- 0.29446   

                   64                  256  
2   1.65640 +- 0.30709  2.84510 +- 0.36700  
8   1.84955 +- 0.33855  3.34521 +- 0.44527  
32  2.25000 +- 0.33076  4.76433 +- 0.43793  
64  2.76334 +- 0.35111  7.65794 +- 0.33436  


In [22]:
print(df)

         1         4         16        64        256
2    1.24957  1.290408  1.358988  1.727423  2.845311
5   1.252761  1.304439  1.365731   1.70738  3.135045
25  1.285223  1.329705  1.497394  2.161159  4.386694
50  1.291652  1.409523  1.762562  2.530653  6.138439
